In [1]:
import pandas as pd
import plotly.graph_objects as go
from gerar_mapa import BASE

# ============================================================
# CONFIGURAÇÃO
# ============================================================

EMPRESAS = ["bram", "starnav", "cbo"]

FUNDO = "#0B1F3A"
BRANCO = "#FFFFFF"

# ============================================================
# PROCESSAMENTO
# ============================================================

for EMPRESA in EMPRESAS:

    ARQUIVO = BASE / "dados" / f"SURVEY_{EMPRESA}_D2.xlsx"

    # --------------------------------------------------------
    # LÊ TODAS AS ABAS
    # --------------------------------------------------------

    abas = pd.read_excel( ARQUIVO, sheet_name=None )

    # --------------------------------------------------------
    # PERCORRE TODAS AS EMBARCAÇÕES
    # --------------------------------------------------------

    for nome_aba, df in abas.items():

        print(f"Processando: {nome_aba}")

        # ====================================================
        # VALIDAÇÃO
        # ====================================================

        if "data_consulta" not in df.columns:
            print(f"Aba {nome_aba} ignorada.")
            continue

        if "data_reportada" not in df.columns:
            print(f"Aba {nome_aba} não possui data_reportada.")
            continue

        if "distancia_km" not in df.columns:
            print(f"Aba {nome_aba} não possui distancia_km.")
            continue

        # ====================================================
        # PREPARAÇÃO DOS DADOS
        # ====================================================

        df["data_consulta"] = pd.to_datetime(
            df["data_consulta"],
            dayfirst=True,
            errors="coerce"
        )

        df["data_reportada"] = pd.to_datetime(
            df["data_reportada"],
            dayfirst=True,
            errors="coerce"
        )

        df["distancia_km"] = pd.to_numeric(
            df["distancia_km"],
            errors="coerce"
        )

        df = df.dropna(
            subset=[
                "data_consulta",
                "data_reportada",
                "distancia_km"
            ]
        )

        # Ordena cronologicamente
        df = df.sort_values(
            "data_reportada"
        ).reset_index(drop=True)

        # ====================================================
        # DISTÂNCIA ACUMULADA
        # ====================================================

        df["distancia_acumulada_km"] = (
            df["distancia_km"].cumsum()
        )

        # ====================================================
        # GRÁFICO
        # ====================================================

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=df["data_consulta"],
                y=df["distancia_acumulada_km"],

                mode="lines+markers",

                name="Distância acumulada",

                # Linha branca
                line=dict(
                    color=BRANCO,
                    width=2
                ),

                # Marcadores brancos
                marker=dict(
                    color=BRANCO,
                    size=5
                ),

                hovertemplate=(
                    "<b>Data:</b> %{x|%d/%m/%Y %H:%M}"
                    "<br>"
                    "<b>Distância acumulada:</b> %{y:,.2f} km"
                    "<extra></extra>"
                )
            )
        )

        # ====================================================
        # CONFIGURAÇÃO DOS EIXOS
        # ====================================================

        fig.update_xaxes(

            title=dict(
                text="Data Reportada",
                font=dict(
                    color=BRANCO
                )
            ),

            tickformat="%d/%m/%Y",

            tickangle=90,

            tickfont=dict(
                color=BRANCO
            ),

            linecolor=BRANCO,
            gridcolor="rgba(255,255,255,0.15)",

            zerolinecolor="rgba(255,255,255,0.2)"
        )

        fig.update_yaxes(

            title=dict(
                text="Distância acumulada (km)",
                font=dict(
                    color=BRANCO
                )
            ),

            tickfont=dict(
                color=BRANCO
            ),

            linecolor=BRANCO,
            gridcolor="rgba(255,255,255,0.15)",

            zerolinecolor="rgba(255,255,255,0.2)"
        )

        # ====================================================
        # LAYOUT
        # ====================================================

        fig.update_layout(

            # Fundo externo
            paper_bgcolor=FUNDO,

            # Fundo interno do gráfico
            plot_bgcolor=FUNDO,

            # Texto geral
            font=dict(
                color=BRANCO,
                family="Arial"
            ),

            # Título
            title=dict(
                text=f"Distância acumulada - {nome_aba}",
                font=dict(
                    color=BRANCO,
                    size=20
                ),
                x=0.5,
                xanchor="center"
            ),

            # Remove legenda desnecessária
            showlegend=False,

            # Ocupa toda a área disponível
            autosize=True,

            # Margens reduzidas
            margin=dict(
                l=70,
                r=25,
                t=60,
                b=80
            ),

            # Hover
            hovermode="x unified",

            hoverlabel=dict(
                bgcolor=FUNDO,
                font=dict(
                    color=BRANCO
                ),
                bordercolor=BRANCO
            )
        )

        # ====================================================
        # CONFIGURAÇÃO DE RESPONSIVIDADE
        # ====================================================

        config = {
            "responsive": True,
            "displayModeBar": True,
            "displaylogo": False
        }

        # ====================================================
        # NOME DO ARQUIVO
        # ====================================================

        arquivo = (
            nome_aba
            .lower()
            .replace(" ", "_")
        )

        # ====================================================
        # DIRETÓRIO DE SAÍDA
        # ====================================================

        saida =  BASE  / "graficos" / "distancia_acumulada" / EMPRESA

        saida.mkdir(
            parents=True,
            exist_ok=True
        )

        # ====================================================
        # EXPORTAÇÃO
        # ====================================================

        fig.write_html(
            saida / f"{arquivo}.html",

            include_plotlyjs="cdn",

            full_html=False,

            config=config,

            # Remove o tamanho fixo do gráfico
            # e deixa o container controlar a dimensão.
            default_width="100%",
            default_height="100%"
        )

        print(
            f"Gráfico salvo: "
            f"{saida / f'{arquivo}.html'}"
        )

    print(
        f"Todos os gráficos da empresa "
        f"{EMPRESA.upper()} foram gerados."
    )
    
    

Processando: BRAM ATLAS
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\bram\bram_atlas.html
Processando: BRAM BAHIA
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\bram\bram_bahia.html
Processando: BRAM BELEM
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\bram\bram_belem.html
Processando: BRAM BRASIL
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\bram\bram_brasil.html
Processando: BRAM BRASILIA
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\bram\bram_brasilia.html
Processando: BRAM BRAVO
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e

Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\cbo\cbo_rio.html
Processando: CBO SUPPORTER
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\cbo\cbo_supporter.html
Processando: CBO VITORIA
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\cbo\cbo_vitoria.html
Processando: CBO WAVE
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\cbo\cbo_wave.html
Processando: CBO WISER
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\distancia_acumulada\cbo\cbo_wiser.html
Processando: DELTA CARDINAL
Gráfico salvo: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica off